# 20. RL policy objectives — REINFORCE, PPO, DPO, GRPO

REINFORCE, PPO, DPO, GRPO의 objective를 tensor 연산으로 직접 계산한다. GRPO는 completion 내부에서 valid token 평균을 먼저 구한 뒤 completion/group 평균을 취한다.


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)


## 1. REINFORCE sequence score-function objective


In [ ]:
token_logp = torch.tensor([[-0.5, -0.7, -0.3, -0.4]], device=device, requires_grad=True)
completion_mask = torch.tensor([[1.0, 1.0, 1.0, 0.0]], device=device)
reward = torch.tensor([1.5], device=device)
sequence_logp = (token_logp * completion_mask).sum(dim=-1)
reinforce_loss = -(reward * sequence_logp).mean()
reinforce_loss.backward()
assert token_logp.grad[0, 3].item() == 0.0
print("REINFORCE loss:", reinforce_loss.item())


## 2. PPO token-level clipped surrogate


In [ ]:
old_logp = torch.tensor([[-0.8, -0.6, -0.7, -0.2], [-0.4, -0.9, -0.5, -0.3]], device=device)
new_logp = torch.tensor([[-0.7, -0.8, -0.6, -0.1], [-0.5, -0.7, -0.6, -0.4]], device=device)
mask = torch.tensor([[1.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0]], device=device)
advantage = torch.tensor([1.0, -0.5], device=device)[:, None]
ratio = torch.exp(new_logp - old_logp)
clipped_ratio = ratio.clamp(0.8, 1.2)
surrogate = torch.minimum(ratio * advantage, clipped_ratio * advantage)
ppo_loss = -(surrogate * mask).sum() / mask.sum()
print("PPO loss:", ppo_loss.item())


## 3. Sampled-action reference-policy KL estimator


In [ ]:
policy_logp = torch.tensor([[-0.6, -0.4, -0.8]], device=device)
reference_logp = torch.tensor([[-0.7, -0.5, -0.6]], device=device)
log_ratio_ref_over_policy = reference_logp - policy_logp
reference_kl = torch.exp(log_ratio_ref_over_policy) - log_ratio_ref_over_policy - 1
assert (reference_kl >= 0).all()
print("reference KL estimator:", reference_kl)


## 4. DPO chosen/rejected sequence margins relative to reference


In [ ]:
policy_chosen = torch.tensor([-3.0, -2.4], device=device)
policy_rejected = torch.tensor([-4.1, -2.9], device=device)
reference_chosen = torch.tensor([-3.3, -2.5], device=device)
reference_rejected = torch.tensor([-3.8, -3.0], device=device)
beta = 0.1
policy_margin = policy_chosen - policy_rejected
reference_margin = reference_chosen - reference_rejected
relative_margin = policy_margin - reference_margin
dpo_loss = -F.logsigmoid(beta * relative_margin).mean()
print("DPO loss:", dpo_loss.item())


## 5. GRPO group-relative advantages


In [ ]:
rewards = torch.tensor([[1.2, 0.2, 0.8, 2.0], [0.1, 0.4, 0.3, -0.2]], device=device)
group_mean = rewards.mean(dim=1, keepdim=True)
group_std = rewards.std(dim=1, keepdim=True, unbiased=False)
group_advantage = (rewards - group_mean) / (group_std + 1e-6)
assert torch.allclose(group_advantage.mean(dim=1), torch.zeros(2, device=device), atol=1e-5)
print("GRPO group advantages:", group_advantage)


## 6. GRPO clipped token objective + reference KL + completion-wise reduction

각 completion은 자기 valid token들의 평균 objective를 하나의 값으로 만들고, 마지막에 completion들을 동일 가중치로 평균한다.


In [ ]:
num_prompts = 2
group_size = 4
max_tokens = 5
old_logp = -torch.rand(num_prompts, group_size, max_tokens, device=device)
current_logp = old_logp + 0.15 * torch.randn_like(old_logp)
reference_logp = old_logp + 0.10 * torch.randn_like(old_logp)
completion_mask = torch.tensor([[[1,1,1,1,0],[1,1,1,0,0],[1,1,1,1,1],[1,1,0,0,0]],[[1,1,1,0,0],[1,1,1,1,0],[1,1,0,0,0],[1,1,1,1,1]]], dtype=torch.float32, device=device)
token_ratio = torch.exp(current_logp - old_logp)
clipped_ratio = token_ratio.clamp(0.8, 1.2)
advantage_per_token = group_advantage[:, :, None]
policy_surrogate = torch.minimum(token_ratio * advantage_per_token, clipped_ratio * advantage_per_token)
ref_over_policy = reference_logp - current_logp
reference_kl = torch.exp(ref_over_policy) - ref_over_policy - 1
kl_beta = 0.04
token_objective = policy_surrogate - kl_beta * reference_kl
token_count_per_completion = completion_mask.sum(dim=-1).clamp_min(1.0)
completion_objective = (token_objective * completion_mask).sum(dim=-1) / token_count_per_completion
grpo_loss = -completion_objective.mean()
clip_indicator = ((token_ratio < 0.8) | (token_ratio > 1.2)).float()
completion_clip_fraction = (clip_indicator * completion_mask).sum(dim=-1) / token_count_per_completion
clip_fraction = completion_clip_fraction.mean()
print("GRPO loss:", grpo_loss.item())
print("completion objectives:", completion_objective)
print("clip fraction:", clip_fraction.item())


## 7. Equal-completion-weight example

두 completion의 길이가 달라도 per-token objective가 같으면 completion 평균값은 동일하다.


In [ ]:
test_objective = torch.tensor([[2.0,2.0,0.0,0.0,0.0],[2.0,2.0,2.0,2.0,2.0]], device=device)
test_mask = torch.tensor([[1.0,1.0,0.0,0.0,0.0],[1.0,1.0,1.0,1.0,1.0]], device=device)
per_completion = (test_objective * test_mask).sum(dim=-1) / test_mask.sum(dim=-1)
assert torch.allclose(per_completion, torch.tensor([2.0, 2.0], device=device))
print("equal completion weights:", per_completion)
